# Step-by-step Bayesian regularization tutorial

This notebook is the transparent teaching version of the two-dimensional dijet regularization study. It intentionally uses the sequential PyROOT/RooUnfold implementation and never calls the compiled C++ parallel scan. The default ensemble is small enough for classroom inspection; increase it only after every intermediate distribution is understood.

The workflow keeps three objects conceptually separate:

1. response-training MC, which supplies migration, purity, efficiency, and the Bayesian prior;
2. a selectable toy truth, which is the known answer;
3. trigger-stitched data, which supply the MB exposure and relative statistical precision.

For every displayed ratio, numerator and denominator are ordinary weighted populations and ROOT's independent-error propagation is used.

## 1. Environment and reusable helpers

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import math
import os
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'CMakeLists.txt').is_file()
                     and (p / 'hist_analysis').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the jetAnalysis repository root')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root, load_roounfold
ROOT = load_root(batch=True)
load_roounfold(ROOT, project_root=PROJECT_ROOT)
import RooUnfold
ROOT.TH1.AddDirectory(False)
ROOT.gStyle.SetOptStat(0)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import DIJET_PTAVE_BINS, STANDARD_DIJET_ETA_CUT_INDEX
from hist_analysis.python.data_trigger_stitching import load_trigger_stitching_config
from hist_analysis.python.histogram_io import load_histogram, resolve_combined_file, resolve_data_file
from hist_analysis.python.plotting import draw_closure
from hist_analysis.python.unfolding import (
    FlattenedBinning, UnfoldingInputKeys, apply_efficiency_correction,
    apply_purity_correction, as_pt_intervals, build_roounfold_response,
    calculate_response_diagnostics, copy_histogram_relative_errors,
    extract_eta_block, flatten_pt_eta_projections, flatten_sparse_response,
    flattened_pt_range_integral, forward_fold_truth, load_unfolding_inputs,
    make_gaussian_toys, prepare_factorized_corrections, project_eta_by_pt,
    scan_bayes_regularization, unfold_bayes,
)
from hist_analysis.python.unfolding_plots import draw_regularization_metrics

## 2. Configuration

`N_TOYS` and `MAX_ITERATIONS` are deliberately modest. `DISPLAY_TOY_INDEX` selects one pseudo-experiment that will be followed through every correction. `DISPLAY_PT_INDEX` selects one pTave block for readable eta overlays; the regularization metric still uses the complete flattened distribution.

In [ ]:
RESPONSE_GENERATOR = 'embedding'
RESPONSE_DIRECTION = 'combined'
RESPONSE_FILE_STEM = 'jetId'
DATA_SELECTION = 'jetId'
TOY_TRUTH_MODEL = 'mixed_prior'  # mixed_prior or cp5
CP5_TOY_TRUTH_FILE = Path(os.environ.get(
    'CP5_TOY_TRUTH_FILE',
    Path.home() / 'cernbox/ana/pPb8160/pythia/cp5tune/HiForestMiniAOD_hist.root'))
CP5_TOY_TRUTH_HISTOGRAM = 'hGenDijetPtEtaCM'
ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
ETA_CUT = 1.9
ETA_RANGE = (-ETA_CUT, ETA_CUT)
PTAVE_INTERVALS = as_pt_intervals(DIJET_PTAVE_BINS)
N_TOYS = 50
MAX_ITERATIONS = 6
RANDOM_SEED = 12345
DISPLAY_TOY_INDEX = 0
DISPLAY_PT_INDEX = 5
RESPONSE_SCALE = 1.0e12
RESAMPLE_NEGATIVE_TOY_BIN = True

TRIGGER_BY_INTERVAL = {
    (50.0, 60.0): 'MinimumBias', (60.0, 70.0): 'MinimumBias',
    (70.0, 80.0): 'MinimumBias', (80.0, 90.0): 'Jet60',
    (90.0, 100.0): 'Jet60', (100.0, 110.0): 'Jet80',
    (110.0, 120.0): 'Jet80', (120.0, 130.0): 'Jet100',
    (130.0, 140.0): 'Jet100', (140.0, 150.0): 'Jet100',
    (150.0, 160.0): 'Jet100', (160.0, 180.0): 'Jet100',
    (180.0, 200.0): 'Jet100', (200.0, 250.0): 'Jet100',
    (250.0, 300.0): 'Jet100', (300.0, 500.0): 'Jet100',
    (500.0, 1000.0): 'Jet100',
}
if TOY_TRUTH_MODEL not in {'mixed_prior', 'cp5'}:
    raise ValueError("TOY_TRUTH_MODEL must be 'mixed_prior' or 'cp5'")
if not 0 <= DISPLAY_TOY_INDEX < N_TOYS:
    raise IndexError('DISPLAY_TOY_INDEX is outside the toy ensemble')
if not 0 <= DISPLAY_PT_INDEX < len(PTAVE_INTERVALS):
    raise IndexError('DISPLAY_PT_INDEX is outside PTAVE_INTERVALS')

STITCHING_DIR = (PROJECT_ROOT / 'hist_analysis/output/data_trigger_stitching'
                 / RESPONSE_DIRECTION / DATA_SELECTION / 'dijet_cm')
candidates = sorted(STITCHING_DIR.glob('trigger_stitching_weights_*.json'))
compatible = []
for candidate in candidates:
    try:
        compatible.append(load_trigger_stitching_config(
            candidate, direction=RESPONSE_DIRECTION, selection=DATA_SELECTION,
            observable='dijet', frame='cm', eta_range=ETA_RANGE,
            eta_cut_index=ETA_CUT_INDEX))
    except ValueError:
        pass
if len(compatible) != 1:
    raise RuntimeError(f'Expected one compatible stitching artifact, found {len(compatible)}')
STITCHING_CONFIG = compatible[0]
mb_records = [r for r in STITCHING_CONFIG.metadata['scales']
              if r.get('reference') in {'MB', 'MinimumBias'}]
if len(mb_records) != 1:
    raise RuntimeError('Expected one direct MB stitching record')
MB_NORMALIZATION_RANGE = tuple(float(x) for x in mb_records[0]['pt_range'])
OUTPUT_DIR = PROJECT_ROOT / 'hist_analysis/output/unfold2D_regularization_tutorial'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Sequential RooUnfold tutorial: compiled scan disabled')
print(f'MB normalization range: {MB_NORMALIZATION_RANGE}')

## 3. Construct the response and inspect the two truth roles

The migration matrix is flattened only for RooUnfold. The displayed eta slice compares the response-training prior with the independently selected toy truth before any detector effects. Independent unit normalization is used only for this shape display; the histograms used later remain unnormalized.

In [ ]:
response_file = resolve_combined_file(BASE_DIR, RESPONSE_GENERATOR, RESPONSE_FILE_STEM)
keys = UnfoldingInputKeys(
    truth=f'hGenDijetPtEtaCM_{ETA_CUT_INDEX}',
    measured=f'hRecoDijetPtEtaCMJerDefExtraUnfold_{ETA_CUT_INDEX}',
    response=f'hGenDijetPtEtaCMVsRecoJerDefExtraPtEtaCM_{ETA_CUT_INDEX}',
    miss=f'hGenDijetPtEtaCMMissJerDefExtra_{ETA_CUT_INDEX}',
    fake=f'hRecoDijetPtEtaCMFakeJerDefExtra_{ETA_CUT_INDEX}')
inputs = load_unfolding_inputs(response_file, keys)
if TOY_TRUTH_MODEL == 'mixed_prior':
    toy_truth_2d = load_histogram(str(response_file), f'hGenDijetPtEtaCMMixedPrior_{ETA_CUT_INDEX}')
else:
    toy_truth_2d = load_histogram(str(CP5_TOY_TRUTH_FILE), CP5_TOY_TRUTH_HISTOGRAM)

project = lambda h, prefix: project_eta_by_pt(
    h, PTAVE_INTERVALS, name_prefix=prefix, eta_range=ETA_RANGE)
training_truth_by_pt = project(inputs.truth, 'hTutorialTrainingTruth')
training_reco_by_pt = project(inputs.measured, 'hTutorialTrainingReco')
miss_by_pt = project(inputs.miss, 'hTutorialMiss')
fake_by_pt = project(inputs.fake, 'hTutorialFake')
toy_truth_by_pt = project(toy_truth_2d, 'hTutorialToyTruth')
training_truth, layout = flatten_pt_eta_projections(
    training_truth_by_pt, name='hTutorialTrainingTruthFlat', pt_bins=PTAVE_INTERVALS)
training_reco, _ = flatten_pt_eta_projections(
    training_reco_by_pt, name='hTutorialTrainingRecoFlat', layout=layout)
training_miss, _ = flatten_pt_eta_projections(miss_by_pt, name='hTutorialMissFlat', layout=layout)
training_fake, _ = flatten_pt_eta_projections(fake_by_pt, name='hTutorialFakeFlat', layout=layout)
toy_truth, _ = flatten_pt_eta_projections(toy_truth_by_pt, name='hTutorialToyTruthFlat', layout=layout)
response_matrix, _ = flatten_sparse_response(
    inputs.response, PTAVE_INTERVALS, name='hTutorialResponse',
    layout=layout, eta_range=ETA_RANGE)
diagnostics = calculate_response_diagnostics(
    response_matrix, training_truth, training_reco,
    explicit_miss=training_miss, explicit_fake=training_fake)
inclusive_response = build_roounfold_response(
    RooUnfold, training_truth, training_reco, response_matrix,
    diagnostics=diagnostics, scale=RESPONSE_SCALE, require_fakes=True,
    name='tutorialInclusiveResponse')
corrections = prepare_factorized_corrections(
    training_reco, training_reco, diagnostics.matched_measured,
    training_truth, diagnostics.matched_truth, name_prefix='hTutorial')
matched_response = build_roounfold_response(
    RooUnfold, diagnostics.matched_truth, diagnostics.matched_measured,
    response_matrix, scale=RESPONSE_SCALE, require_fakes=False,
    name='tutorialMatchedResponse')

pt_low, pt_high = PTAVE_INTERVALS[DISPLAY_PT_INDEX]
prior_shape = training_truth_by_pt[DISPLAY_PT_INDEX].Clone('hTutorialPriorShape')
truth_shape = toy_truth_by_pt[DISPLAY_PT_INDEX].Clone('hTutorialTruthShape')
for h in (prior_shape, truth_shape):
    h.SetDirectory(0)
    h.Scale(1.0 / h.Integral())
prior_canvas, prior_ratios = draw_closure(
    {'Response-training prior': prior_shape, 'Toy truth': truth_shape},
    nominal='Response-training prior',
    title=f'{pt_low:g} < p_{{T}}^{{ave}} < {pt_high:g} GeV',
    x_title='#eta_{CM}', y_title='Unit-normalized entries',
    ratio_range=(0.5, 1.5), draw_nominal_ratio=False,
    output=OUTPUT_DIR / '01_prior_vs_toy_truth.pdf',
    canvas_name='canvas_tutorial_prior_truth')
prior_canvas

## 4. Stitch data and inspect purity and efficiency

Each pTave interval takes its established trigger after that trigger is scaled to the MB exposure. Purity converts inclusive reconstructed yields to matched reconstructed yields. Efficiency later converts matched truth to inclusive truth.

In [ ]:
data_maps = {}
data_by_pt = []
for pt_index, interval in enumerate(PTAVE_INTERVALS):
    trigger = TRIGGER_BY_INTERVAL[interval]
    if trigger not in data_maps:
        filename = resolve_data_file(BASE_DIR / 'exp', trigger, RESPONSE_DIRECTION, DATA_SELECTION)
        histogram = load_histogram(str(filename), STITCHING_CONFIG.histogram_keys[trigger])
        histogram.Scale(STITCHING_CONFIG.weights[trigger])
        data_maps[trigger] = histogram
    data_by_pt.append(project_eta_by_pt(
        data_maps[trigger], (interval,), name_prefix=f'hTutorialData{pt_index}',
        eta_range=ETA_RANGE)[0])
data_reco, _ = flatten_pt_eta_projections(
    data_by_pt, name='hTutorialStitchedData', layout=layout)

purity_eta = extract_eta_block(corrections.purity, training_reco_by_pt[DISPLAY_PT_INDEX],
                               layout, DISPLAY_PT_INDEX, name='hTutorialPurityEta')
efficiency_eta = extract_eta_block(corrections.efficiency, training_truth_by_pt[DISPLAY_PT_INDEX],
                                   layout, DISPLAY_PT_INDEX, name='hTutorialEfficiencyEta')
correction_canvas = ROOT.TCanvas('canvas_tutorial_corrections', '', 900, 450)
correction_canvas.Divide(2, 1)
correction_canvas.cd(1); purity_eta.SetTitle('Reco purity;#eta_{CM};Matched reco / inclusive reco'); purity_eta.SetMinimum(0); purity_eta.SetMaximum(1.1); purity_eta.Draw('E1')
correction_canvas.cd(2); efficiency_eta.SetTitle('Truth efficiency;#eta_{CM};Matched truth / inclusive truth'); efficiency_eta.SetMinimum(0); efficiency_eta.SetMaximum(1.1); efficiency_eta.Draw('E1')
correction_canvas.Update()
correction_canvas.Print(str(OUTPUT_DIR / '02_purity_efficiency.pdf'))
correction_canvas._objects = (purity_eta, efficiency_eta)
correction_canvas

## 5. Apply one global MinimumBias normalization and forward-fold

First forward-fold the unscaled toy truth. The ratio of MB data to folded-toy yield in the direct MB stitching overlap supplies one global exposure factor. Scaling the complete truth by one number preserves every pTave and eta shape. The scaled truth is then forward-folded again.

In [ ]:
unscaled_toy_truth = toy_truth.Clone('hTutorialUnscaledToyTruth')
unscaled_toy_truth.SetDirectory(0)
unscaled_fold = forward_fold_truth(
    inclusive_response, unscaled_toy_truth, diagnostics=diagnostics,
    add_fakes=True, fake_normalization='matched_fraction',
    name='hTutorialUnscaledFold')
mb_projection = project_eta_by_pt(
    data_maps['MinimumBias'], (MB_NORMALIZATION_RANGE,),
    name_prefix='hTutorialMBNormalization', eta_range=ETA_RANGE)[0]
mb_yield = mb_projection.Integral()
folded_yield = flattened_pt_range_integral(
    unscaled_fold.histogram, layout, MB_NORMALIZATION_RANGE)
mb_scale = mb_yield / folded_yield
toy_truth.Scale(mb_scale)
target_fold = forward_fold_truth(
    inclusive_response, toy_truth, diagnostics=diagnostics,
    add_fakes=True, fake_normalization='matched_fraction',
    name='hTutorialForwardFoldedTruth')
toy_expectation = copy_histogram_relative_errors(
    target_fold.histogram, data_reco, name='hTutorialToyExpectation',
    source_zero_relative_tolerance=1.0e-14, layout=layout,
    use_nearest_reference_for_empty=True)
print(f'MB data / unscaled folded toy = {mb_yield:g} / {folded_yield:g}')
print(f'One global truth scale factor = {mb_scale:.10g}')

expectation_eta = extract_eta_block(
    toy_expectation, training_reco_by_pt[DISPLAY_PT_INDEX], layout,
    DISPLAY_PT_INDEX, name='hTutorialExpectationEta')
data_eta = data_by_pt[DISPLAY_PT_INDEX].Clone('hTutorialDataEtaDisplay')
data_eta.SetDirectory(0)
fold_canvas, fold_ratios = draw_closure(
    {'Stitched data': data_eta, 'Forward-folded toy expectation': expectation_eta},
    nominal='Stitched data',
    title=f'{pt_low:g} < p_{{T}}^{{ave}} < {pt_high:g} GeV',
    x_title='#eta_{CM}', y_title='Reco entries', ratio_range=(0.5, 1.5),
    draw_nominal_ratio=False, output=OUTPUT_DIR / '03_folded_expectation_vs_data.pdf',
    canvas_name='canvas_tutorial_fold_data')
fold_canvas

## 6. Generate toys and follow one toy through the purity correction

All iterations reuse the same toy ensemble. This makes iteration-to-iteration comparisons paired. The first overlay checks the selected fluctuated toy against its expectation. The second shows exactly how the fixed purity correction changes that toy before unfolding.

In [ ]:
toys = make_gaussian_toys(
    toy_expectation, n_toys=N_TOYS, random_seed=RANDOM_SEED, layout=layout,
    stop_on_negative=False, resample_negative=RESAMPLE_NEGATIVE_TOY_BIN,
    smear_zero_content=False)
signal_toys = [apply_purity_correction(
    toy, corrections.purity, name=f'hTutorialSignalToy{i}')
    for i, toy in enumerate(toys)]
display_toy_eta = extract_eta_block(
    toys[DISPLAY_TOY_INDEX], training_reco_by_pt[DISPLAY_PT_INDEX], layout,
    DISPLAY_PT_INDEX, name='hTutorialDisplayToyEta')
display_signal_eta = extract_eta_block(
    signal_toys[DISPLAY_TOY_INDEX], training_reco_by_pt[DISPLAY_PT_INDEX], layout,
    DISPLAY_PT_INDEX, name='hTutorialDisplaySignalEta')
toy_canvas, toy_ratios = draw_closure(
    {'Expectation': expectation_eta, 'Fluctuated toy': display_toy_eta},
    nominal='Expectation', title=f'Toy {DISPLAY_TOY_INDEX}',
    x_title='#eta_{CM}', y_title='Reco entries', ratio_range=(0.5, 1.5),
    draw_nominal_ratio=False, output=OUTPUT_DIR / '04_toy_vs_expectation.pdf',
    canvas_name='canvas_tutorial_toy')
purity_canvas, purity_ratios = draw_closure(
    {'Inclusive reco toy': display_toy_eta, 'Purity-corrected matched toy': display_signal_eta},
    nominal='Inclusive reco toy', title=f'Toy {DISPLAY_TOY_INDEX}',
    x_title='#eta_{CM}', y_title='Reco entries', ratio_range=(0.5, 1.1),
    draw_nominal_ratio=False, output=OUTPUT_DIR / '05_purity_correction.pdf',
    canvas_name='canvas_tutorial_purity_step')
display(toy_canvas); display(purity_canvas)

## 7. Unfold the selected toy one iteration at a time

This is intentionally explicit. Every iteration makes a fresh sequential `RooUnfoldBayes` call, and every result is divided by efficiency before comparison with the known inclusive truth. The upper panel shows the distributions; the lower panel shows unfolded/truth.

In [ ]:
truth_eta = extract_eta_block(
    toy_truth, training_truth_by_pt[DISPLAY_PT_INDEX], layout,
    DISPLAY_PT_INDEX, name='hTutorialScaledTruthEta')
iteration_distributions = {'Toy truth': truth_eta}
unfolded_display_toys = {}
for iterations in range(1, MAX_ITERATIONS + 1):
    matched = unfold_bayes(
        RooUnfold, matched_response, signal_toys[DISPLAY_TOY_INDEX],
        iterations=iterations, handle_fakes=False, calculate_covariance=False,
        suppress_native_output=True, name=f'hTutorialToyUnfoldedMatchedIter{iterations}')
    inclusive = matched.histogram.Clone(f'hTutorialToyUnfoldedIter{iterations}')
    inclusive.SetDirectory(0)
    for global_bin in range(1, layout.n_global_bins + 1):
        efficiency = corrections.efficiency.GetBinContent(global_bin)
        inclusive.SetBinContent(global_bin, inclusive.GetBinContent(global_bin) / efficiency)
        inclusive.SetBinError(global_bin, inclusive.GetBinError(global_bin) / efficiency)
    unfolded_display_toys[iterations] = inclusive
    iteration_distributions[f'{iterations} iteration' if iterations == 1 else f'{iterations} iterations'] = extract_eta_block(
        inclusive, training_truth_by_pt[DISPLAY_PT_INDEX], layout, DISPLAY_PT_INDEX,
        name=f'hTutorialUnfoldedEtaIter{iterations}')
iteration_canvas, iteration_ratios = draw_closure(
    iteration_distributions, nominal='Toy truth',
    title=f'Toy {DISPLAY_TOY_INDEX}, {pt_low:g} < p_{{T}}^{{ave}} < {pt_high:g} GeV',
    x_title='#eta_{CM}', y_title='Truth entries', ratio_range=(0.7, 1.3),
    draw_nominal_ratio=False, output=OUTPUT_DIR / '06_all_iterations_vs_truth.pdf',
    canvas_name='canvas_tutorial_iterations')
iteration_canvas

## 8. Sequential ensemble scan: bias, variance, and MSE

Now repeat the same operation for every toy. No C++ moment kernel or worker pool is used. For truth bin j and iteration n, the notebook calculates the ensemble mean, squared bias, unbiased variance, and MSE = bias squared + variance. Absolute metrics retain yield-squared units; relative metrics divide each bin by its truth squared.

In [ ]:
def progress(iterations, maximum):
    print(f'Sequential RooUnfold iteration {iterations}/{maximum}')

scan = scan_bayes_regularization(
    RooUnfold, matched_response, signal_toys, toy_truth,
    max_iterations=MAX_ITERATIONS, selection_mode='both',
    handle_fakes=False, calculate_covariance=False,
    efficiency=corrections.efficiency, progress=progress)
print(f'Absolute-MSE minimum: {scan.absolute_selected_iterations}')
print(f'Relative-MSE minimum: {scan.relative_selected_iterations}')
absolute_canvas, absolute_objects = draw_regularization_metrics(
    scan, mode='absolute', output=OUTPUT_DIR / '07_absolute_metrics.pdf',
    log_y=True)
relative_canvas, relative_objects = draw_regularization_metrics(
    scan, mode='relative', output=OUTPUT_DIR / '08_relative_metrics.pdf',
    log_y=True)
display(absolute_canvas); display(relative_canvas)

## 9. Compare the ensemble mean at every iteration

A single toy can fluctuate atypically. This final plot instead compares the mean unfolded ensemble at every iteration with the known truth in the selected pTave block. Read it together with the bias-variance plots: improving closure can be offset by increasing variance.

In [ ]:
mean_distributions = {'Toy truth': truth_eta}
for metric in scan.metrics:
    label = f'{metric.iterations} iteration' if metric.iterations == 1 else f'{metric.iterations} iterations'
    mean_distributions[label] = extract_eta_block(
        metric.mean, training_truth_by_pt[DISPLAY_PT_INDEX], layout,
        DISPLAY_PT_INDEX, name=f'hTutorialMeanEtaIter{metric.iterations}')
mean_canvas, mean_ratios = draw_closure(
    mean_distributions, nominal='Toy truth',
    title=f'Ensemble mean, {pt_low:g} < p_{{T}}^{{ave}} < {pt_high:g} GeV',
    x_title='#eta_{CM}', y_title='Truth entries', ratio_range=(0.9, 1.1),
    draw_nominal_ratio=False, output=OUTPUT_DIR / '09_mean_iterations_vs_truth.pdf',
    canvas_name='canvas_tutorial_mean_iterations')
mean_canvas

## 10. Interpretation checklist

Before quoting an iteration count, answer these questions from the plots:

- Is the toy truth visibly different from the response-training prior?
- Does forward folding give a physically reasonable reconstructed shape?
- Are purity and efficiency smooth and nonzero?
- Does the selected toy behave typically compared with the ensemble mean?
- Does bias squared decrease before variance dominates?
- Do absolute and relative MSE prefer the same iteration?
- Is the preferred point an interior minimum rather than a scan boundary?

The answer is conditional on the response, toy-truth model, binning, and uncertainty model. Repeat the exercise with CP5 and other controlled truth deformations before adopting a production value.